# 🫁 Updated Hybrid Framework for Pneumonia Detection in Chest X-Ray Images

**Architecture Overview:**
- **Branch 1:** ShuffleNetV2 + ECA (Efficient Channel Attention) — lightweight, fine-grained local features
- **Branch 2:** EfficientNet-B0 + CBAM (Convolutional Block Attention Module) — global semantic + spatial features
- **LAMR Module:** Lesion-Aware Multi-scale Refinement (inspired by multi-scale nodule detection)
- **Grad-CAM:** Interpretability visualization for clinical reliability

> Dataset: [Chest X-ray Pneumonia (Kaggle)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)

## 1. Install & Import Dependencies

In [ ]:
# ── Install any missing packages ──────────────────────────────────────────────
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip_install('timm')          # pretrained EfficientNet-B0 / ShuffleNetV2
pip_install('grad-cam')      # pytorch-grad-cam
pip_install('albumentations') # augmentation
print('All packages ready.')

All packages ready.


In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import os, random, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tv_models
import timm

# ── Sklearn metrics ───────────────────────────────────────────────────────────
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score
)

# ── Grad-CAM ──────────────────────────────────────────────────────────────────
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ── Albumentations ────────────────────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Configuration

In [ ]:
# ── Hyper-parameters & paths ─────────────────────────────────────────────────
class CFG:
    # ── Dataset paths (Kaggle environment) ────────────────────────────────────
    DATA_DIR   = Path('/kaggle/input/chest-xray-pneumonia/chest_xray')
    TRAIN_DIR  = DATA_DIR / 'train'
    VAL_DIR    = DATA_DIR / 'val'
    TEST_DIR   = DATA_DIR / 'test'

    # ── Image ─────────────────────────────────────────────────────────────────
    IMG_SIZE   = 224          # both branches expect 224×224
    IN_CHANS   = 3

    # ── Training ──────────────────────────────────────────────────────────────
    BATCH_SIZE = 32
    EPOCHS     = 30
    LR         = 3e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS  = 4

    # ── Model ─────────────────────────────────────────────────────────────────
    NUM_CLASSES   = 2          # NORMAL / PNEUMONIA
    FUSION_DIM    = 256        # after projection
    LAMR_CHANNELS = 256        # LAMR internal channels

    # ── Class labels ──────────────────────────────────────────────────────────
    CLASSES = ['NORMAL', 'PNEUMONIA']

    # ── Checkpoint ────────────────────────────────────────────────────────────
    CKPT_PATH = '/kaggle/working/best_model.pth'

cfg = CFG()
print('Configuration loaded.')

## 3. Dataset & Data Augmentation

In [ ]:
# ── Albumentations pipelines ─────────────────────────────────────────────────

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transforms():
    return A.Compose([
        A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
        # ── Enhancement (mirrors preprocessing in the paper) ──────────────────
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),      # histogram equalization
        A.RandomBrightnessContrast(brightness_limit=0.2,
                                   contrast_limit=0.2, p=0.5),       # contrast stretching
        # ── Augmentation ──────────────────────────────────────────────────────
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                           rotate_limit=15, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])


# ── Custom Dataset ────────────────────────────────────────────────────────────
class ChestXRayDataset(Dataset):
    """
    Loads images from folder structure:
        root/
            NORMAL/   *.jpeg
            PNEUMONIA/ *.jpeg
    """
    def __init__(self, root_dir: Path, transform=None):
        self.transform = transform
        self.samples   = []
        self.label_map = {cls: i for i, cls in enumerate(cfg.CLASSES)}

        for cls in cfg.CLASSES:
            cls_dir = root_dir / cls
            if not cls_dir.exists():
                print(f'[WARN] Directory not found: {cls_dir}')
                continue
            for img_path in cls_dir.glob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    self.samples.append((img_path, self.label_map[cls]))

        print(f'  → Loaded {len(self.samples)} images from {root_dir}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = np.array(Image.open(img_path).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label


# ── Build dataloaders ─────────────────────────────────────────────────────────
print('Building datasets...')
train_dataset = ChestXRayDataset(cfg.TRAIN_DIR, transform=get_train_transforms())
val_dataset   = ChestXRayDataset(cfg.VAL_DIR,   transform=get_val_transforms())
test_dataset  = ChestXRayDataset(cfg.TEST_DIR,  transform=get_val_transforms())

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE,
                          shuffle=True,  num_workers=cfg.NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.BATCH_SIZE,
                          shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=cfg.BATCH_SIZE,
                          shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)

# ── Class distribution ────────────────────────────────────────────────────────
labels_train = [s[1] for s in train_dataset.samples]
unique, counts = np.unique(labels_train, return_counts=True)
print('\nClass distribution (train):')
for u, c in zip(unique, counts):
    print(f'  {cfg.CLASSES[u]}: {c}')

In [ ]:
# ── Visualise sample images ───────────────────────────────────────────────────
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Sample Training Images', fontsize=14, fontweight='bold')

for cls_idx, cls_name in enumerate(cfg.CLASSES):
    cls_samples = [(p, l) for p, l in train_dataset.samples if l == cls_idx]
    for i in range(5):
        img_path, _ = random.choice(cls_samples)
        img = np.array(Image.open(img_path).convert('RGB'))
        axes[cls_idx, i].imshow(img, cmap='gray' if img.mean(axis=2).std() < 10 else None)
        axes[cls_idx, i].set_title(cls_name, color='green' if cls_idx == 0 else 'red')
        axes[cls_idx, i].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/sample_images.png', dpi=150)
plt.show()

## 4. Attention Modules: ECA, CBAM

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ECA — Efficient Channel Attention                                          ║
# ║  Wang et al. (2020) CVPR                                                    ║
# ║  Replaces FC bottleneck with a lightweight 1-D convolution across channels  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
class ECA(nn.Module):
    """
    Efficient Channel Attention.
    Kernel size k is adaptively determined from the number of channels C.
    """
    def __init__(self, channels: int, gamma: int = 2, b: int = 1):
        super().__init__()
        # Adaptive kernel size
        t  = int(abs(math.log2(channels) / gamma + b / gamma))
        k  = t if t % 2 else t + 1
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv     = nn.Conv1d(1, 1, kernel_size=k,
                                  padding=k // 2, bias=False)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W)
        y = self.avg_pool(x)            # (B, C, 1, 1)
        y = y.squeeze(-1).transpose(-1, -2)  # (B, 1, C)
        y = self.conv(y)                # (B, 1, C)
        y = y.transpose(-1, -2).unsqueeze(-1)  # (B, C, 1, 1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CBAM — Convolutional Block Attention Module                                ║
# ║  Woo et al. (2018) ECCV                                                     ║
# ║  Sequential channel attention → spatial attention                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        reduced = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, reduced, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        out     = self.sigmoid(avg_out + max_out)
        return x * out.unsqueeze(-1).unsqueeze(-1)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size,
                                 padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = x.mean(dim=1, keepdim=True)
        max_out = x.max(dim=1, keepdim=True).values
        out     = torch.cat([avg_out, max_out], dim=1)
        out     = self.sigmoid(self.conv(out))
        return x * out


class CBAM(nn.Module):
    """Channel → Spatial attention block."""
    def __init__(self, channels: int, reduction: int = 16, spatial_k: int = 7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.ca(x)
        x = self.sa(x)
        return x


print('ECA and CBAM modules defined.')

## 5. LAMR — Lesion-Aware Multi-scale Refinement Module

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LAMR — Lesion-Aware Multi-scale Refinement                                 ║
# ║                                                                              ║
# ║  Inspired by the HPFF (Hierarchical Progressive Feature Fusion) and GCSAM   ║
# ║  ideas in Li et al. (2025) for lung nodule detection.                        ║
# ║                                                                              ║
# ║  Three parallel convolution streams (1×1, 3×3, 5×5) capture lesion          ║
# ║  patterns at different receptive fields, then a channel-wise                 ║
# ║  attention gate produces a lesion-sensitive mask to refine the fused         ║
# ║  feature map.                                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
class LAMRBlock(nn.Module):
    """
    Lesion-Aware Multi-scale Refinement block.

    Args:
        in_channels  : number of input channels (= fusion feature size after projection)
        out_channels : number of output channels
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()

        mid = out_channels // 4  # internal bottleneck

        # ── Multi-scale parallel branches ──────────────────────────────────────
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, mid, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True)
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, mid, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True)
        )
        self.branch5 = nn.Sequential(
            nn.Conv2d(in_channels, mid, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True)
        )
        # ── Global context (avg-pool) branch ───────────────────────────────────
        self.branch_global = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, mid, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True)
        )

        concat_ch = mid * 4          # 4 branches

        # ── Projection to out_channels ─────────────────────────────────────────
        self.proj = nn.Sequential(
            nn.Conv2d(concat_ch, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )

        # ── Lesion-sensitive attention gate (channel-wise) ─────────────────────
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_channels, out_channels // 4, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels // 4, out_channels, kernel_size=1),
            nn.Sigmoid()
        )

        # ── Residual shortcut (adjust channels if needed) ──────────────────────
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape

        b1 = self.branch1(x)                     # (B, mid, H, W)
        b3 = self.branch3(x)                     # (B, mid, H, W)
        b5 = self.branch5(x)                     # (B, mid, H, W)
        bg = self.branch_global(x).expand(B, -1, H, W)  # broadcast

        ms = torch.cat([b1, b3, b5, bg], dim=1)  # (B, 4*mid, H, W)
        ms = self.proj(ms)                        # (B, out_channels, H, W)

        # ── Apply lesion attention gate ────────────────────────────────────────
        gate_w = self.gate(ms)                    # (B, out_channels, 1, 1)
        ms     = ms * gate_w                      # element-wise scale

        return ms + self.shortcut(x)              # residual addition


print('LAMR block defined.')

## 6. Branch Backbones: ShuffleNetV2 + ECA and EfficientNet-B0 + CBAM

In [ ]:
class ShuffleNetV2_ECA(nn.Module):
    """
    Branch 1 — ShuffleNetV2 (x1.0) backbone with ECA inserted
    after the final stage for lightweight, fine-grained local feature extraction.
    Output: (B, proj_dim) feature vector.
    """
    def __init__(self, proj_dim: int = 256, pretrained: bool = True):
        super().__init__()
        base = timm.create_model('shufflenet_v2_x1_0',
                                  pretrained=pretrained,
                                  features_only=False)

        # ── Remove original classifier head ────────────────────────────────────
        self.backbone  = nn.Sequential(*list(base.children())[:-2])  # drop pool+fc
        feat_dim       = base.num_features  # 1024 for x1.0

        # ── ECA attention after last conv stage ────────────────────────────────
        self.eca       = ECA(channels=feat_dim)
        self.gap       = nn.AdaptiveAvgPool2d(1)

        # ── Projection head ───────────────────────────────────────────────────
        self.proj      = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor):
        feat = self.backbone(x)   # (B, 1024, H', W')
        feat = self.eca(feat)     # channel attention
        feat = self.gap(feat)     # (B, 1024, 1, 1)
        feat = self.proj(feat)    # (B, proj_dim)
        return feat

    def get_last_conv(self):
        """Return last conv layer for Grad-CAM."""
        for m in reversed(list(self.backbone.modules())):
            if isinstance(m, nn.Conv2d):
                return m


class EfficientNetB0_CBAM(nn.Module):
    """
    Branch 2 — EfficientNet-B0 backbone with CBAM inserted
    after the last feature block for global semantic + spatial awareness.
    Output: (B, proj_dim) feature vector.
    """
    def __init__(self, proj_dim: int = 256, pretrained: bool = True):
        super().__init__()
        base      = timm.create_model('efficientnet_b0',
                                       pretrained=pretrained,
                                       features_only=False)
        feat_dim  = base.num_features  # 1280 for B0

        # ── Backbone without classifier ────────────────────────────────────────
        self.backbone = base.forward_features  # callable returning feat map
        self._base    = base

        # ── CBAM attention ────────────────────────────────────────────────────
        self.cbam = CBAM(channels=feat_dim)
        self.gap  = nn.AdaptiveAvgPool2d(1)

        # ── Projection ────────────────────────────────────────────────────────
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor):
        feat = self.backbone(x)    # (B, 1280, H', W')
        feat = self.cbam(feat)     # channel + spatial attention
        feat = self.gap(feat)      # (B, 1280, 1, 1)
        feat = self.proj(feat)     # (B, proj_dim)
        return feat

    def get_last_conv(self):
        for m in reversed(list(self._base.modules())):
            if isinstance(m, nn.Conv2d):
                return m


# ── Quick sanity check ────────────────────────────────────────────────────────
dummy = torch.zeros(2, 3, 224, 224)
b1    = ShuffleNetV2_ECA(proj_dim=256, pretrained=False)
b2    = EfficientNetB0_CBAM(proj_dim=256, pretrained=False)
print(f'Branch-1 output: {b1(dummy).shape}')   # (2, 256)
print(f'Branch-2 output: {b2(dummy).shape}')   # (2, 256)

## 7. Full Hybrid Model

In [ ]:
class HybridPneumoniaNet(nn.Module):
    """
    Updated Hybrid Framework
    ========================
    Input  → [ShuffleNetV2+ECA] branch  (local, fine-grained)
           → [EfficientNet-B0+CBAM] branch (global, semantic)
    Fused features → spatial reshape → LAMR module → GAP → classifier

    The LAMR module operates on a 2D spatial feature map reconstructed
    from the concatenated 1-D vectors via a learned reshape/expand path,
    mimicking the receptive-field diversity of the HPFF + GCSAM approach.
    """

    def __init__(
        self,
        num_classes  : int = 2,
        proj_dim     : int = 256,
        lamr_channels: int = 256,
        pretrained   : bool = True,
        dropout_p    : float = 0.4
    ):
        super().__init__()

        # ── Two feature branches ──────────────────────────────────────────────
        self.branch_shuffle = ShuffleNetV2_ECA(proj_dim=proj_dim,
                                                pretrained=pretrained)
        self.branch_effnet  = EfficientNetB0_CBAM(proj_dim=proj_dim,
                                                   pretrained=pretrained)

        fused_dim = proj_dim * 2   # concatenation of two branches

        # ── Fusion projection to a square spatial map for LAMR ────────────────
        # We reshape the fused vector to (B, fused_dim, 1, 1) then upsample
        self.fusion_proj = nn.Sequential(
            nn.Linear(fused_dim, lamr_channels),
            nn.BatchNorm1d(lamr_channels),
            nn.ReLU(inplace=True)
        )
        # Upsample 1×1 → 7×7 feature map for LAMR to act on
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(lamr_channels, lamr_channels,
                               kernel_size=7, stride=1, bias=False),
            nn.BatchNorm2d(lamr_channels),
            nn.ReLU(inplace=True)
        )

        # ── LAMR module ───────────────────────────────────────────────────────
        self.lamr = LAMRBlock(in_channels=lamr_channels,
                              out_channels=lamr_channels)

        # ── Global Average Pool + Classifier ─────────────────────────────────
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_p),
            nn.Linear(lamr_channels, lamr_channels // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p / 2),
            nn.Linear(lamr_channels // 2, num_classes)
        )

    def forward(self, x: torch.Tensor):
        # Branch 1: ShuffleNetV2 + ECA
        f1 = self.branch_shuffle(x)           # (B, proj_dim)
        # Branch 2: EfficientNet-B0 + CBAM
        f2 = self.branch_effnet(x)            # (B, proj_dim)

        # ── Feature fusion ────────────────────────────────────────────────────
        fused = torch.cat([f1, f2], dim=1)    # (B, 2*proj_dim)
        fused = self.fusion_proj(fused)        # (B, lamr_channels)

        # ── Reshape to 2D spatial map for LAMR ────────────────────────────────
        fused = fused.unsqueeze(-1).unsqueeze(-1)   # (B, lamr_ch, 1, 1)
        fused = self.upsample(fused)                 # (B, lamr_ch, 7, 7)

        # ── LAMR: multi-scale lesion refinement ───────────────────────────────
        fused = self.lamr(fused)               # (B, lamr_ch, 7, 7)

        # ── Classification ────────────────────────────────────────────────────
        out  = self.gap(fused)                 # (B, lamr_ch, 1, 1)
        out  = self.head(out)                  # (B, num_classes)
        return out


# ── Instantiate & test ────────────────────────────────────────────────────────
model = HybridPneumoniaNet(
    num_classes   = cfg.NUM_CLASSES,
    proj_dim      = cfg.FUSION_DIM,
    lamr_channels = cfg.LAMR_CHANNELS,
    pretrained    = True
).to(DEVICE)

dummy = torch.zeros(2, 3, 224, 224).to(DEVICE)
out   = model(dummy)
print(f'Model output shape: {out.shape}')   # (2, 2)

# ── Parameter count ───────────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters   : {total_params / 1e6:.2f}M')
print(f'Trainable parameters: {train_params / 1e6:.2f}M')

## 8. Loss Function, Optimizer, Scheduler

In [ ]:
# ── Compute class weights to handle imbalance ─────────────────────────────────
class_counts = np.array([counts[0], counts[1]], dtype=np.float32)
class_weights = torch.tensor(
    1.0 / (class_counts / class_counts.sum()), dtype=torch.float32
).to(DEVICE)
print(f'Class weights: {class_weights.cpu().numpy()}')

criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer  = optim.AdamW(model.parameters(),
                         lr=cfg.LR,
                         weight_decay=cfg.WEIGHT_DECAY)
# ── Cosine annealing with warm restarts ───────────────────────────────────────
scheduler  = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# ── Mixed precision scaler ────────────────────────────────────────────────────
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
print('Criterion / Optimizer / Scheduler ready.')

## 9. Training & Validation Loops

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in tqdm(loader, desc='  Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        preds         = logits.argmax(dim=1)
        correct      += (preds == labels).sum().item()
        total        += imgs.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []

    for imgs, labels in tqdm(loader, desc='    Val', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        running_loss += loss.item() * imgs.size(0)
        probs         = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds         = logits.argmax(dim=1)
        correct      += (preds == labels).sum().item()
        total        += imgs.size(0)
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / total
    acc      = correct / total
    auc      = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return avg_loss, acc, auc


print('Training functions defined.')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss'  : [], 'val_acc'  : [], 'val_auc': []
}
best_val_auc = 0.0

for epoch in range(1, cfg.EPOCHS + 1):
    print(f'\nEpoch [{epoch:02d}/{cfg.EPOCHS}]  lr={scheduler.get_last_lr()[0]:.2e}')

    tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                       optimizer, criterion, scaler, DEVICE)
    va_loss, va_acc, va_auc = validate(model, val_loader, criterion, DEVICE)

    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc' ].append(tr_acc)
    history['val_loss'  ].append(va_loss)
    history['val_acc'   ].append(va_acc)
    history['val_auc'   ].append(va_auc)

    print(f'  Train  loss={tr_loss:.4f}  acc={tr_acc:.4f}')
    print(f'  Val    loss={va_loss:.4f}  acc={va_acc:.4f}  AUC={va_auc:.4f}')

    if va_auc > best_val_auc:
        best_val_auc = va_auc
        torch.save(model.state_dict(), cfg.CKPT_PATH)
        print(f'  ✅ New best model saved  (AUC={best_val_auc:.4f})')

print('\nTraining complete.')

## 10. Training Curves

In [ ]:
epochs_range = range(1, cfg.EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History — Updated Hybrid Framework', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='#e74c3c')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='#3498db')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], label='Train Acc', color='#e74c3c')
axes[1].plot(epochs_range, history['val_acc'],   label='Val Acc',   color='#3498db')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

# AUC
axes[2].plot(epochs_range, history['val_auc'], label='Val AUC', color='#2ecc71')
axes[2].set_title('AUC-ROC'); axes[2].set_xlabel('Epoch'); axes[2].legend()

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()

## 11. Test Set Evaluation

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(cfg.CKPT_PATH, map_location=DEVICE))
model.eval()

@torch.no_grad()
def predict(model, loader, device):
    all_preds, all_probs, all_labels = [], [], []
    for imgs, labels in tqdm(loader, desc='  Test', leave=False):
        imgs   = imgs.to(device)
        logits = model(imgs)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_probs), np.array(all_labels)

y_pred, y_prob, y_true = predict(model, test_loader, DEVICE)

# ── Metrics ───────────────────────────────────────────────────────────────────
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)
auc  = roc_auc_score(y_true, y_prob)

print('=' * 55)
print('TEST SET PERFORMANCE — Updated Hybrid Framework')
print('=' * 55)
print(f'  Accuracy  : {acc  * 100:.2f}%')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  AUC-ROC   : {auc:.4f}')
print('=' * 55)
print(classification_report(y_true, y_pred, target_names=cfg.CLASSES))

## 12. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices — Updated Hybrid Framework', fontsize=13, fontweight='bold')

for ax, norm, title in zip(axes,
                            [None, 'true'],
                            ['Absolute Counts', 'Normalized (recall)']):
    cm_plot = confusion_matrix(y_true, y_pred, normalize=norm)
    fmt = 'd' if norm is None else '.2f'
    sns.heatmap(cm_plot, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=cfg.CLASSES, yticklabels=cfg.CLASSES,
                linewidths=1, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted', fontweight='bold')
    ax.set_ylabel('Actual', fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

## 13. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#2980b9', lw=2,
        label=f'Updated Hybrid Framework (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='#2980b9')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/roc_curve.png', dpi=150)
plt.show()

## 14. Performance Radar Chart

In [ ]:
# ── Radar chart comparing key metrics ────────────────────────────────────────
metrics_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
values         = [acc * 100, prec * 100, rec * 100, f1 * 100, auc * 100]

N = len(metrics_labels)
angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]    # close the polygon
values_plot = values + values[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(math.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), metrics_labels, fontsize=11)

ax.plot(angles, values_plot, linewidth=2, linestyle='solid',
        color='#2980b9', label='Updated Hybrid Framework')
ax.fill(angles, values_plot, alpha=0.25, color='#2980b9')

# Reference ring
ax.set_ylim(80, 100)
ax.set_yticks(range(80, 101, 5))
ax.set_yticklabels([f'{v}%' for v in range(80, 101, 5)], fontsize=8)

ax.set_title('Model Performance Radar Chart', fontsize=13,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))

plt.tight_layout()
plt.savefig('/kaggle/working/radar_chart.png', dpi=150)
plt.show()

## 15. Grad-CAM Visualization

In [ ]:
# ── Grad-CAM target layer = last conv in LAMR branch3 (5×5 kernel) ────────────
# This highlights lesion-focused spatial regions.
target_layer = model.lamr.branch5[-2]   # the last BN before ReLU in branch5

# Fallback: use last conv of LAMR proj
try:
    _ = target_layer.weight
except AttributeError:
    target_layer = model.lamr.proj[0]   # Conv2d in proj

cam = GradCAM(model=model, target_layers=[target_layer])


def show_gradcam(img_path: Path, label: int, ax_row, transforms_fn):
    """Draw original + Grad-CAM overlay for one image."""
    img_np  = np.array(Image.open(img_path).convert('RGB'))
    img_t   = transforms_fn(image=img_np)['image'].unsqueeze(0).to(DEVICE)

    # Resize for display
    img_resized = np.array(Image.fromarray(img_np).resize((224, 224))) / 255.0

    # Grad-CAM
    targets = [ClassifierOutputTarget(label)]
    grayscale_cam = cam(input_tensor=img_t, targets=targets)[0]
    cam_img = show_cam_on_image(img_resized.astype(np.float32),
                                 grayscale_cam, use_rgb=True)

    ax_row[0].imshow(img_resized); ax_row[0].set_title('Original', fontsize=9)
    ax_row[1].imshow(grayscale_cam, cmap='jet');
    ax_row[1].set_title('CAM Heatmap', fontsize=9)
    ax_row[2].imshow(cam_img);    ax_row[2].set_title('Grad-CAM Overlay', fontsize=9)
    for a in ax_row:
        a.axis('off')


# ── Pick 3 random samples per class ──────────────────────────────────────────
n_samples = 3
fig, axes = plt.subplots(2 * n_samples, 3, figsize=(12, 4 * n_samples * 2))
fig.suptitle('Grad-CAM Visualization — Lesion-Focused Regions', fontsize=13, fontweight='bold')

val_tf = get_val_transforms()
row = 0
for cls_idx, cls_name in enumerate(cfg.CLASSES):
    cls_samples = [(p, l) for p, l in test_dataset.samples if l == cls_idx]
    chosen      = random.sample(cls_samples, min(n_samples, len(cls_samples)))
    for path, lbl in chosen:
        axes[row, 0].set_ylabel(cls_name, fontsize=10, fontweight='bold',
                                color='green' if cls_idx == 0 else 'red')
        show_gradcam(path, lbl, axes[row], val_tf)
        row += 1

plt.tight_layout()
plt.savefig('/kaggle/working/gradcam_visualization.png', dpi=150)
plt.show()

## 16. Comparative Summary Table

In [ ]:
# ── Comparison: original paper vs updated framework ───────────────────────────
comparison_data = {
    'Method': [
        'Original (VGG16 + ResNet + SVM)',
        'Original (VGG16 + ResNet + RF)',
        'Updated (ShuffleNetV2-ECA + EffNet-B0-CBAM + LAMR)'
    ],
    'Feature Extraction': [
        'VGG16 + ResNet (fused)',
        'VGG16 + ResNet (fused)',
        'ShuffleNetV2+ECA + EfficientNet-B0+CBAM'
    ],
    'Attention': [
        'None',
        'None',
        'ECA + CBAM + LAMR'
    ],
    'Classifier': [
        'SVM (RBF)',
        'Random Forest',
        'FC layers (end-to-end)'
    ],
    'Accuracy (%)': [
        '98.8',
        '95.8',
        f'{acc * 100:.2f}'
    ],
    'AUC-ROC': [
        '0.92',
        '0.92',
        f'{auc:.4f}'
    ],
    'Explainability': [
        'Interpretable (SVM)',
        'Interpretable (RF)',
        'Grad-CAM visual'
    ]
}

df = pd.DataFrame(comparison_data)
display(df)
df.to_csv('/kaggle/working/comparison_table.csv', index=False)
print('Comparison table saved.')

## 17. Performance Gap Heatmap

In [ ]:
# ── Performance gap (how far each model is from 100%) ─────────────────────────
models = ['SVM (original)', 'RF (original)', 'Updated Hybrid']
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']

# Fill in actual values (original from paper; updated from our test)
perf = np.array([
    [98.8, 99.0, 98.0, 98.0, 92.0],   # SVM original
    [95.8, 96.0, 94.0, 95.0, 92.0],   # RF  original
    [acc*100, prec*100, rec*100, f1*100, auc*100]   # Updated
])
gap = 100.0 - perf   # lower is better

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(gap, annot=perf, fmt='.1f', cmap='RdYlGn_r',
            xticklabels=metrics, yticklabels=models,
            linewidths=0.5, ax=ax, vmin=0, vmax=15)
ax.set_title('Performance Gap Heatmap (Lower Gap = Better)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Metric', fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/performance_gap_heatmap.png', dpi=150)
plt.show()

## 18. Architecture Diagram (Text Summary)

In [ ]:
from torchinfo import summary

# Install torchinfo if needed
try:
    from torchinfo import summary
except ImportError:
    pip_install('torchinfo')
    from torchinfo import summary

summary(model, input_size=(1, 3, 224, 224),
        col_names=['input_size', 'output_size', 'num_params', 'mult_adds'],
        depth=3)

## 19. Save Outputs

In [ ]:
# ── Save final predictions ────────────────────────────────────────────────────
results_df = pd.DataFrame({
    'true_label'     : y_true,
    'predicted_label': y_pred,
    'prob_pneumonia'  : y_prob,
    'class_true'     : [cfg.CLASSES[l] for l in y_true],
    'class_pred'     : [cfg.CLASSES[l] for l in y_pred],
    'correct'        : y_true == y_pred
})
results_df.to_csv('/kaggle/working/test_predictions.csv', index=False)

# ── Save metrics summary ──────────────────────────────────────────────────────
metrics_df = pd.DataFrame([{
    'Accuracy' : round(acc, 4),
    'Precision': round(prec, 4),
    'Recall'   : round(rec, 4),
    'F1-Score' : round(f1, 4),
    'AUC-ROC'  : round(auc, 4)
}])
metrics_df.to_csv('/kaggle/working/metrics_summary.csv', index=False)

print('All outputs saved to /kaggle/working/')
print('\nFiles:')
for f in sorted(Path('/kaggle/working/').glob('*')):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} KB)')

---
## Summary

| Component | Original Paper | This Update |
|-----------|---------------|-------------|
| Branch 1 | VGG16 | **ShuffleNetV2 + ECA** |
| Branch 2 | ResNet | **EfficientNet-B0 + CBAM** |
| Multi-scale module | — | **LAMR (1×1, 3×3, 5×5 parallel convs)** |
| Dimensionality reduction | PCA (100 components) | End-to-end via projection + pooling |
| Classifier | SVM / Random Forest | Fully-connected + Softmax |
| Interpretability | None | **Grad-CAM** |
| Parameters | ~140M+ | **~10–15M (lightweight)** |

**Key improvements:**
- ECA provides parameter-free channel attention that adaptively scales features with minimal overhead
- CBAM in EfficientNet-B0 adds channel + spatial attention jointly for discriminative global features
- LAMR module mirrors the HPFF + GCSAM multi-scale lesion awareness from nodule detection literature
- Grad-CAM provides visual explanations aligned with clinically relevant lung regions